In [1]:
from pathlib import Path
import pandas as pd

def find_project_root():
    path = Path.cwd().resolve()
    for candidate in [path, *path.parents]:
        if (candidate / "data" / "processed").is_dir():
            return candidate
    raise FileNotFoundError("Could not find data/processed from " + str(path))

PROCESSED = find_project_root() / "data" / "processed"
dev = pd.read_csv(PROCESSED / "cohort_2017-2020.csv")
val = pd.read_csv(PROCESSED / "cohort_2021-2023.csv")
Y = "outcome_140_90"

NUMERIC = ["age", "bmi", "waist", "sitting_min_day",
           "hba1c", "total_chol", "hdl", "creatinine"]
CATEGORICAL = ["female", "smoking", "diabetes"]

# 1. Numeric predictors: median in negatives vs positives
print("1. Median by outcome (2017-2020)")
med = dev.groupby(Y)[NUMERIC].median().T
med.columns = ["negative", "positive"]
med["difference"] = med["positive"] - med["negative"]
print(med.round(1))

# 2. Positive rate across quartiles of each numeric predictor
print("\n2. % positive by quartile (Q1 = lowest, 2017-2020)")
rows = {}
for col in NUMERIC:
    # rank first so that many tied values (e.g. sitting = 300) still split into 4 groups
    q = pd.qcut(dev[col].rank(method="first"), 4, labels=["Q1", "Q2", "Q3", "Q4"])
    rows[col] = dev.groupby(q, observed=True)[Y].mean() * 100
print(pd.DataFrame(rows).T.round(1))

# 3. Categorical predictors: % positive per category
print("\n3. % positive by category (2017-2020)")
for col in CATEGORICAL:
    t = dev.groupby(col, dropna=False)[Y].agg(n="size", pct_positive="mean")
    t["pct_positive"] = (t["pct_positive"] * 100).round(1)
    print(f"\n{col}\n{t}")

# 4. Correlation between numeric predictors (redundancy check)
print("\n4. Correlations (Spearman, 2017-2020), only |r| >= 0.5")
corr = dev[NUMERIC].corr(method="spearman")
pairs = [(a, b, corr.loc[a, b]) for i, a in enumerate(NUMERIC)
         for b in NUMERIC[i + 1:] if abs(corr.loc[a, b]) >= 0.5]
for a, b, r in pairs:
    print(f"  {a:<16}{b:<16}{r:6.2f}")
if not pairs:
    print("  none")

# 5. Does the age effect look the same in the validation period?
print("\n5. % positive by age group, both periods")
bins = [20, 40, 60, 81]
labels = ["20-39", "40-59", "60+"]
both = pd.DataFrame({
    "2017-2020": dev.groupby(pd.cut(dev["age"], bins, right=False, labels=labels), observed=True)[Y].mean() * 100,
    "2021-2023": val.groupby(pd.cut(val["age"], bins, right=False, labels=labels), observed=True)[Y].mean() * 100,
})
print(both.round(1))
print("\nAll numbers are unweighted (sample, not US population).")

1. Median by outcome (2017-2020)
                 negative  positive  difference
age                  41.0      60.0        19.0
bmi                  27.7      28.1         0.4
waist                95.3      99.5         4.2
sitting_min_day     300.0     300.0         0.0
hba1c                 5.4       5.7         0.3
total_chol          182.0     195.0        13.0
hdl                  51.0      51.0         0.0
creatinine            0.8       0.9         0.1

2. % positive by quartile (Q1 = lowest, 2017-2020)
                   Q1    Q2    Q3    Q4
age               3.5   7.6  14.9  27.9
bmi              11.2  14.0  12.4  15.2
waist             8.7  11.9  14.9  16.9
sitting_min_day  12.5  13.6  14.6  12.7
hba1c             8.2   9.9  14.4  20.9
total_chol        9.2  11.9  14.1  18.0
hdl              13.6  12.8  13.5  13.3
creatinine        9.9  11.8  13.6  17.5

3. % positive by category (2017-2020)

female
           n  pct_positive
female                    
0       2287          